In [2]:
import os
import gc
import time
import statistics
from dataclasses import dataclass
from typing import Callable, List, Tuple, Dict, Any, Sequence, Union, Optional
from pathlib import Path
from __future__ import annotations

import cv2
import numpy as np
import torch
import onnxruntime as ort

from v1.SSD_from_scratch import mySSD as oldSSD
from v2.model_files.SSD_from_scratch import mySSD as newSSD
from SSDInt8_ONNX_Pred import SSDInt8ONNXPredictor, PreprocessConfig

# desktop, laptop, ubuntu
machine = 'ubuntu'

# Setup path to data folder
if machine == 'laptop':
    folder_path = Path(r"C:\self-driving-car\data")
elif machine == 'desktop':
    folder_path = Path(r"C:\Udacity_car_data\data")
elif machine == 'ubuntu':
    folder_path = Path(r"/mnt/c/Udacity_car_data/data")

train_path = folder_path / "train"
test_path = folder_path / "test"



os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(1)

In [13]:
@dataclass
class BenchStats:
    batch_size: int
    n_runs: int
    median_ms_per_batch: float
    p90_ms_per_batch: float
    p95_ms_per_batch: float
    imgs_per_sec_median: float

def percentile(sorted_xs, p: float) -> float:
    k = (len(sorted_xs) - 1) * p
    f = int(np.floor(k)); c = int(np.ceil(k))
    if f == c: return sorted_xs[f]
    return sorted_xs[f] + (sorted_xs[c] - sorted_xs[f]) * (k - f)

def benchmark_single_image(
    run_fn: Callable[[List[np.ndarray]], Any],
    images: List[np.ndarray],
    warmup: int = 50,
    runs: int = 1000,
) -> List[float]:
    # Each call uses batch=[img], batch size is always 1
    gc.disable()
    try:
        n = len(images)
        # warm-up
        for i in range(min(warmup, runs)):
            _ = run_fn(images[i % n])

        times_s = []
        for i in range(runs):
            img = images[i % n]
            t0 = time.perf_counter_ns()
            _ = run_fn(img)
            t1 = time.perf_counter_ns()
            times_s.append((t1 - t0) * 1e-9)
        return times_s
    finally:
        gc.enable()

def summarize_ms(times_s: List[float]) -> dict:
    xs = sorted([t * 1e3 for t in times_s])  # ms/image
    med = statistics.median(xs)
    return {
        "n": len(xs),
        "median_ms": med,
        "p90_ms": percentile(xs, 0.90),
        "p95_ms": percentile(xs, 0.95),
        "imgs_per_sec_median": 1.0 / (med * 1e-3),
    }

def make_singleton_batches(images, n_batches: int):
    # cycles through images; each "batch" is [img]
    batches = []
    for i in range(n_batches):
        batches.append([images[i % len(images)]])
    return batches

In [ ]:
class PytorchSSDPipeline:
    def __init__(
        self,
        model: oldSSD | newSSD,
        preprocess_fn: Callable[[List[np.ndarray]], torch.Tensor],
        # postprocess_fn: Callable[[Any], Any],
        # num_threads: int = 8,
    ):
        self.model = model.eval()
        # torch.set_num_threads(num_threads)
        # torch.set_num_interop_threads(1)
        self.preprocess_fn = preprocess_fn
        # self.postprocess_fn = postprocess_fn

    def __call__(self, batch_imgs: List[np.ndarray]):
        with torch.inference_mode():
            x = self.preprocess_fn(batch_imgs)   # shape [B, C, H, W] on CPU
            if type(self.model) == newSSD:
                det = self.model.predict(x, score_thresh=0.3, nms_thresh=0.5, iou_variant="DIoU", max_per_img=50)
            else:
                det = self.model.predict(x, score_thresh=0.3, nms_thresh=0.5, max_per_img=50)
            return det




ImgIn = Union[str, np.ndarray, torch.Tensor]
BatchIn = Union[ImgIn, List[ImgIn]]


class OnnxSSDPipeline:
    """
    Uses SSDInt8ONNXPredictor.__call__ so your benchmark includes:
      - numpy preprocess
      - ORT inference
      - Python post formatting (labels->str, scores->list[float], boxes->pixel coords)

    Accepts either:
      - a single image (np.ndarray / torch.Tensor / path str)
      - or a list with length 1 (batch size fixed at 1)

    Input layout:
      - HWC: (H,W,3)
      - CHW: (3,H,W)
    """

    def __init__(
        self,
        *,
        predictor: Optional["SSDInt8ONNXPredictor"] = None,
        onnx_model_path: Optional[str] = None,
        class_to_idx: Optional[Dict[str, int]] = None,
        providers: Optional[Sequence[str]] = None,
        preprocess_cfg: Optional["PreprocessConfig"] = None,
        output_names: Tuple[str, str, str] = ("boxes_out", "scores_out", "labels_out"),
        input_layout: str = "auto",  # "auto" | "hwc" | "chw"
    ):
        if predictor is None:
            if onnx_model_path is None or class_to_idx is None:
                raise ValueError("Provide predictor=... OR (onnx_model_path=..., class_to_idx=...).")
            if preprocess_cfg is None:
                preprocess_cfg = PreprocessConfig()  # defaults to input_color="bgr"
            predictor = SSDInt8ONNXPredictor(
                onnx_model_path=onnx_model_path,
                class_to_idx=class_to_idx,
                providers=providers,
                preprocess_cfg=preprocess_cfg,
                output_names=output_names,
            )

        self.pred = predictor
        self.input_layout = input_layout.lower()
        if self.input_layout not in ("auto", "hwc", "chw"):
            raise ValueError("input_layout must be one of: 'auto', 'hwc', 'chw'.")

    def __call__(self, x: BatchIn) -> Dict[str, Any]:
        # Allow a list input but enforce batch size = 1
        if isinstance(x, list):
            if len(x) != 1:
                raise ValueError(f"Batch size must be 1; got list of length {len(x)}.")
            x = x[0]

        # If user passes a path, load it here so we can guarantee correct color handling.
        # Predictor loads paths via cv2.imread (BGR). If predictor.pre_cfg.input_color == "rgb"
        # and you pass a path directly, it would treat BGR as RGB (wrong).
        if isinstance(x, str):
            img_bgr = cv2.imread(x, cv2.IMREAD_COLOR)
            if img_bgr is None:
                raise ValueError(f"cv2.imread failed for: {x}")
            img = self._match_predictor_expected_color(img_bgr, current_color="bgr")
            return self.pred(img)

        # torch.Tensor -> numpy
        if isinstance(x, torch.Tensor):
            if x.device.type != "cpu":
                x = x.cpu()
            x = x.detach().numpy()

        if not isinstance(x, np.ndarray):
            raise TypeError(f"Unsupported input type: {type(x)}")

        img = self._to_hwc(x)  # ndarray HWC, 3 channels, contiguous
        img = self._match_predictor_expected_color(img, current_color="rgb")  # assume arrays are RGB unless you say otherwise

        return self.pred(img)

    def _to_hwc(self, arr: np.ndarray) -> np.ndarray:
        if arr.ndim != 3:
            raise ValueError(f"Expected 3D image array, got shape {arr.shape}")

        layout = self.input_layout
        if layout == "auto":
            if arr.shape[2] in (3, 4):
                layout = "hwc"
            elif arr.shape[0] in (3, 4):
                layout = "chw"
            else:
                raise ValueError(f"Cannot infer HWC vs CHW from shape {arr.shape}; set input_layout='hwc' or 'chw'.")

        if layout == "chw":
            arr = np.transpose(arr, (1, 2, 0))  # CHW -> HWC

        # Drop alpha if present
        if arr.shape[2] == 4:
            arr = arr[:, :, :3]

        return np.ascontiguousarray(arr)

    def _match_predictor_expected_color(self, img_hwc: np.ndarray, *, current_color: str) -> np.ndarray:
        """
        Ensures the array we pass into SSDInt8ONNXPredictor matches predictor.pre_cfg.input_color.

        - current_color is what *img_hwc currently is* ("bgr" or "rgb")
        - predictor.pre_cfg.input_color is what predictor assumes for ndarray inputs
        """
        want = self.pred.pre_cfg.input_color.lower()
        cur = current_color.lower()
        if want not in ("bgr", "rgb"):
            raise ValueError(f"Predictor preprocess_cfg.input_color must be 'bgr' or 'rgb', got {self.pred.pre_cfg.input_color}")

        if cur == want:
            return img_hwc

        # Convert to the predictor's expected color
        if cur == "bgr" and want == "rgb":
            return cv2.cvtColor(img_hwc, cv2.COLOR_BGR2RGB)
        if cur == "rgb" and want == "bgr":
            return cv2.cvtColor(img_hwc, cv2.COLOR_RGB2BGR)

        raise ValueError(f"Unexpected color conversion cur={cur} -> want={want}")

In [23]:
def run_suite(images: List[np.ndarray], pt_pipe, ort_pipe):
    results = {"pytorch": [], "onnx": []}
    # batches = make_singleton_batches(images, batch_size=bs, n_batches=len(images))

    if pt_pipe is not None:
        t_pt = benchmark_single_image(run_fn=pt_pipe, images=images, warmup=50, runs=500)
        results["pytorch"].append(summarize_ms(t_pt))
    if ort_pipe is not None:
        t_ort = benchmark_single_image(run_fn=ort_pipe, images=images, warmup=50, runs=500)
        results["onnx"].append(summarize_ms(t_ort))

    return results

In [16]:
def load_images_uint8_chw(
    folder: str,
    *,
    exts=(".jpg", ".jpeg", ".png"),
    limit: int | None = None,
    convert_to_rgb: bool = False,   # keep False if your preprocessing expects BGR
) -> list[np.ndarray]:
    folder = Path(folder)
    paths = sorted([p for p in folder.iterdir() if p.suffix.lower() in exts])
    if limit is not None:
        paths = paths[:limit]

    imgs: list[np.ndarray] = []
    for p in paths:
        im = cv2.imread(str(p), cv2.IMREAD_COLOR)  # uint8 HWC, BGR
        if im is None:
            raise ValueError(f"Failed to read: {p}")
        if convert_to_rgb:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        # enforce contiguous memory (avoids occasional stride surprises)
        im = np.ascontiguousarray(np.transpose(im, (2, 0, 1)))
        imgs.append(im)

    if not imgs:
        raise ValueError(f"No images found in {folder} with extensions {exts}")
    return imgs


def preprocess_uint8_chw_rgb(
    x_chw: Union[np.ndarray, torch.Tensor],
    *,
    out_size: Tuple[int, int] = (300, 300),
    mean: Sequence[float] = (0.485, 0.456, 0.406),
    std: Sequence[float]  = (0.229, 0.224, 0.225),
) -> torch.Tensor:
    """
    Input:
        x_chw: uint8 (C,H,W), RGB, values in [0,255]
    Output:
        torch.float32 (1,C,out_H,out_W)

    Equivalent to:
        v2.ToImage()
        v2.ToDtype(torch.float32, scale=True)
        v2.Resize((300,300), antialias=True)
        v2.Normalize(mean=..., std=...)
    """
    # Convert to torch.Tensor
    if isinstance(x_chw, np.ndarray):
        x = torch.from_numpy(x_chw)
    elif isinstance(x_chw, torch.Tensor):
        x = x_chw
    else:
        raise TypeError(f"Expected numpy.ndarray or torch.Tensor, got {type(x_chw)}")

    if x.ndim != 3:
        raise ValueError(f"Expected shape (C,H,W), got {tuple(x.shape)}")
    if x.dtype != torch.uint8:
        raise TypeError(f"Expected dtype uint8, got {x.dtype}")

    # ToDtype(float32, scale=True) for uint8 => /255
    x = x.to(torch.float32) / 255.0

    # Add batch dim: (1,C,H,W)
    x = x.unsqueeze(0)

    # Resize to (300,300) with antialiasing (bilinear like torchvision for tensors)
    try:
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False, antialias=True)
    except TypeError:
        # Fallback if antialias not supported in your PyTorch version
        x = torch.nn.functional.interpolate(x, size=out_size, mode="bilinear", align_corners=False)

    # Normalize
    mean_t = torch.tensor(mean, dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    std_t  = torch.tensor(std,  dtype=torch.float32, device=x.device).view(1, -1, 1, 1)
    x = (x - mean_t) / std_t

    return x

In [21]:
# load images
img_list = np.array(load_images_uint8_chw(folder=test_path, convert_to_rgb=True, limit=1000))

# load old PyTorch model
old_ssd = oldSSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                          in_channels=3,
                          variances=(0.1, 0.2))
WEIGHTS_PATH = r"/mnt/c/Users/eblac/Documents/GitHub/self-driving-car/app_files/saved_models/noZoomOut_Bootstrap.pth"
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
old_ssd.load_state_dict(state_dict, strict=False)
old_ssd.to(device='cpu');

# load PyTorch new SSD model
new_ssd = newSSD(class_to_idx_dict={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                          in_channels=3,
                          variances=(0.1, 0.2))
WEIGHTS_PATH = Path.home() / "repos" / "automotive-ssd-object-detection" / "v2" / "saved_models" / "DIoU_mAP_551_iou_thresh_45_max_img_per_det_200.pth"
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
new_ssd.load_state_dict(state_dict, strict=False)
new_ssd.to(device='cpu');

# need to use these settings w/ .predict:
# score_threshold=0.30,
# iou_threshold=0.50,
# max_output_boxes_per_class=50,

In [47]:
PytorchSSDPipeline(model=ssd_model_noZO_BS, preprocess_fn=preprocess_uint8_chw_rgb)(img_list[0])

[{'labels': tensor([1, 1, 1, 2]),
  'scores': tensor([0.8854, 0.6741, 0.4775, 0.3346]),
  'boxes': tensor([[149.2532, 142.2011, 157.8897, 154.2081],
          [126.9213, 143.8637, 133.4680, 153.3449],
          [133.1610, 143.1958, 140.0436, 153.8862],
          [272.5420, 121.0653, 287.7605, 183.8527]])}]

In [48]:
OnnxSSDPipeline(predictor=None,
                onnx_model_path=r"C:\Users\eblac\Documents\GitHub\self-driving-car\PTQ_testing\ssd_int8_with_pre_post.onnx",
                class_to_idx={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                providers=None,
                preprocess_cfg=PreprocessConfig(input_color="rgb"))(img_list[0])

{'labels': ['car', 'car', 'car', 'pedestrian'],
 'scores': [0.9376913905143738,
  0.7002373933792114,
  0.4642316401004791,
  0.40516844391822815],
 'boxes': array([[255.83345, 242.15436, 269.64023, 261.37155],
        [215.73732, 245.20201, 226.39445, 260.8237 ],
        [228.58606, 244.49791, 239.2432 , 261.52783],
        [464.7621 , 208.87772, 489.20007, 313.0607 ]], dtype=float32)}

In [13]:
print("----- Old SSD and Old int8 ONNX -----")
run_suite(images=img_list,
          pt_pipe=PytorchSSDPipeline(model=old_ssd, preprocess_fn=preprocess_uint8_chw_rgb),
          ort_pipe=OnnxSSDPipeline(predictor=None,
                onnx_model_path=Path.home() / "repos" / "automotive-ssd-object-detection" / "PTQ_testing" / "ssd_int8_with_pre_post.onnx",
                class_to_idx={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                providers=None,
                preprocess_cfg=PreprocessConfig(input_color="rgb"))
        )

----- Old SSD and Old int8 ONNX -----


{'pytorch': [{'n': 500,
   'median_ms': 214.83971200000002,
   'p90_ms': 231.953103,
   'p95_ms': 238.0228826,
   'imgs_per_sec_median': 4.654632938625425}],
 'onnx': [{'n': 500,
   'median_ms': 42.5262515,
   'p90_ms': 45.421441800000004,
   'p95_ms': 46.19402835,
   'imgs_per_sec_median': 23.514887033953602}]}

In [14]:
print("----- New SSD and Old int8 ONNX -----")
run_suite(images=img_list,
          pt_pipe=PytorchSSDPipeline(model=new_ssd, preprocess_fn=preprocess_uint8_chw_rgb),
          ort_pipe=OnnxSSDPipeline(predictor=None,
                onnx_model_path=Path.home() / "repos" / "automotive-ssd-object-detection" / "PTQ_testing" / "ssd_int8_with_pre_post.onnx",
                class_to_idx={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                providers=None,
                preprocess_cfg=PreprocessConfig(input_color="rgb"))
        )

----- New SSD and Old int8 ONNX -----


{'pytorch': [{'n': 500,
   'median_ms': 219.4449055,
   'p90_ms': 225.0235557,
   'p95_ms': 226.2686043,
   'imgs_per_sec_median': 4.556952451101673}],
 'onnx': [{'n': 500,
   'median_ms': 43.43674750000001,
   'p90_ms': 47.1590563,
   'p95_ms': 48.19751965,
   'imgs_per_sec_median': 23.021981560658972}]}

In [26]:
print("----- Old int8 ONNX w/ pre+post -----")
run_suite(images=img_list,
          pt_pipe=None,
          ort_pipe=OnnxSSDPipeline(predictor=None,
                onnx_model_path=Path.home() / "repos" / "automotive-ssd-object-detection" / "PTQ_testing" / "ssd_int8_with_pre_post.onnx",
                class_to_idx={'biker': 0, 'car': 1, 'pedestrian': 2, 'trafficLight': 3, 'truck': 4},
                providers=None,
                preprocess_cfg=PreprocessConfig(input_color="rgb"))
        )

----- Old int8 ONNX w/ pre+post -----


{'pytorch': [],
 'onnx': [{'n': 500,
   'median_ms': 34.669529999999995,
   'p90_ms': 37.6221446,
   'p95_ms': 38.44116455,
   'imgs_per_sec_median': 28.843771461568707}]}